# BFF Camera Demo - Training notebook (Google Colab)

Trains the YOLOX-style bucket detector + 5-class fill classifier against
the labeled dataset in this Colab session, exports ONNX, and (optionally)
uploads the artifacts to Supabase Storage `models-public/`.

Usage:
1. Open in Colab (free tier is enough for 1-2 epoch smoke).
2. Fill the cell below with credentials (or rely on the BFF Settings UI
   that this notebook writes to).
3. Run all cells.

In [ ]:
# 0. Environment setup - clone the four repos over SSH, install deps.
# The repos are private, so the clone goes through SSH.
# An SSH key that already exists in this runtime is reused as-is; nothing is
# ever deleted. Only when no key exists is one generated, and then it is
# registered with GitHub through the API (token scope: admin:public_key).
from google.colab import userdata
import os
import subprocess
from pathlib import Path

ORG = 'mahardhika'
REF = 'v0.1.0'          # same tag in all four repos
REPOS = {
    'bff-core':     f'git@github.com:{ORG}/bff-core.git',
    'bff-backend':  f'git@github.com:{ORG}/bff-backend.git',
    'bff-frontend': f'git@github.com:{ORG}/bff-frontend.git',
    'bff-ml':       f'git@github.com:{ORG}/bff-ml.git',
}

SSH_DIR = Path.home() / '.ssh'
SSH_DIR.mkdir(parents=True, exist_ok=True)
os.chmod(SSH_DIR, 0o700)

# Reuse an existing key - never remove one.
key = next((p for p in (SSH_DIR / 'id_ed25519', SSH_DIR / 'id_rsa')
            if p.exists()), None)
if key is not None:
    print('using existing key:', key)
else:
    key = SSH_DIR / 'id_ed25519'
    subprocess.run(['ssh-keygen', '-t', 'ed25519', '-N', '', '-C', 'colab-bff',
                    '-f', str(key)], check=True)
    pub = (SSH_DIR / (key.name + '.pub')).read_text().strip()
    try:
        import requests
        token = userdata.get('GITHUB_TOKEN')
        resp = requests.post(
            'https://api.github.com/user/keys',
            headers={
                'Authorization': f'Bearer {token}',
                'Accept': 'application/vnd.github+json',
            },
            json={'title': 'colab-bff', 'key': pub},
            timeout=30,
        )
        print('register key:', resp.status_code, resp.text[:200])
    except Exception as exc:  # noqa: BLE001
        print('could not register the key automatically:', exc)
        print('add this public key in GitHub -> Settings -> SSH keys:')
        print(pub)

# Trust the GitHub host key (appended, existing entries are kept).
known_hosts = SSH_DIR / 'known_hosts'
with known_hosts.open('a') as handle:
    subprocess.run(['ssh-keyscan', '-H', 'github.com'], stdout=handle, check=False)
os.chmod(known_hosts, 0o600)

ssh_env = {**os.environ, 'GIT_SSH_COMMAND': 'ssh -o StrictHostKeyChecking=accept-new'}
for name, url in REPOS.items():
    subprocess.run(
        ['git', 'clone', '--branch', REF, '--depth', '1', url, f'/content/{name}'],
        env=ssh_env,
        check=True,
    )

# Install order matters: bff-core first - bff-backend and bff-ml depend on it
# and it is not published on PyPI.
%pip install -e "/content/bff-core"
%pip install -e "/content/bff-backend[cuda,cpu]"
%pip install -e "/content/bff-ml[cuda,dev]"


In [ ]:
# 1. Credentials - filled in by the user.
import os
from getpass import getpass

os.environ['SUPABASE_URL'] = os.environ.get('SUPABASE_URL', 'https://naltduhojaazpodznvry.supabase.co')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass('Supabase service role key: ')
os.environ['S3_ENDPOINT_URL'] = os.environ.get('S3_ENDPOINT_URL', 'https://naltduhojaazpodznvry.supabase.co/storage/v1/s3')
os.environ['S3_REGION'] = os.environ.get('S3_REGION', 'ap-southeast-1')
os.environ['S3_ACCESS_KEY_ID'] = getpass('S3 access key: ')
os.environ['S3_SECRET_ACCESS_KEY'] = getpass('S3 secret key: ')
os.environ['INFERENCE_PROVIDER'] = 'cuda'

In [ ]:
# 2. Verify the GPU is reachable.
import torch
print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi

In [ ]:
# 3. Mount Drive (where the labeled dataset lives).
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/bff-data'  # adjust
import os; os.makedirs(DATA_DIR, exist_ok=True)
print(f'data dir: {DATA_DIR}')
print('contents:', os.listdir(DATA_DIR)[:10])

In [ ]:
# 4. Run training. Edit args as needed.
# DATA_DIR must point at a COCO root: annotations.json + images/.
%cd /content/bff-ml
!python -m ml train \
    --data-source=local \
    --data-path=$DATA_DIR/detector \
    --kind=detector \
    --model-name=bucket-detector \
    --version=0.1.0 \
    --epochs=50 \
    --batch-size=16 \
    --lr=1e-3 \
    --device=cuda \
    --no-sync


In [ ]:
# 4b. Fill classifier (5 classes: empty/partial/full/over/not_visible).
%cd /content/bff-ml
!python -m ml train \
    --data-source=local \
    --data-path=$DATA_DIR/classifier \
    --kind=classifier \
    --model-name=fill-classifier \
    --version=0.1.0 \
    --epochs=120 \
    --batch-size=16 \
    --lr=1e-3 \
    --device=cuda \
    --no-sync


In [ ]:
# 5. Inspect the latest run, then sync to Supabase if it looks good.
import os, glob
runs = sorted(glob.glob('/content/bff-ml/runs/*'), key=os.path.getmtime)
latest = runs[-1]
print('latest run:', latest)
print('files:', os.listdir(latest))
%cd /content/bff-ml
!python -m ml sync --run-dir={latest} --name=bucket-detector --version=0.1.0 --kind=detector


After the sync, the new weights are at `s3://models-public/bucket-detector/0.1.0/detector.onnx`
and the BE will pick them up on its next 5-minute refresh (or on the
next `/healthz` if you set `MODEL_REFRESH_INTERVAL_S=60`).